# Notebook 3: Real Data Validation

Validate NB1/NB2 findings on real financial and non-financial data.

## Key Design Principle

NB1 showed that **rich data-specific factual text causes w ≈ 0.91** (model trusts Chronos), while **generic factual lowers w to ≈ 0.79** (opens the text channel). NB2 confirmed this: accurate `describe_series()` factual text produced w ≈ 0.93–0.97 across all 48 runs, suppressing text influence. The empty prompt (P5) produced the largest shift.

**Implication for NB3:** To properly test whether predictive signals can steer the forecast on real data, we must use **generic factual sections** to lower w and open the text channel. We then compare generic vs rich factual to quantify the factual bottleneck on real data.

## Experiments

| Exp | Question | Data | Adaptation from NB1/NB2 |
|-----|----------|------|-------------------------|
| **3A** | Does generic factual unlock text influence on real financial data? | S&P 500, Gold, BTC | Test {Rich, Generic} × {Bull, Bear, Neutral} |
| **3B** | With the bottleneck removed, can predictive signals steer direction? | S&P 500, Gold, BTC | Use **only** generic factual, compare directional forecasts |
| **3C** | Does the original cement confound replicate? | Cement (same SKU) | Rerun original experiment with confound fixed |
| **3D** | Does real-data volatility affect text sensitivity? | 3 cement SKUs (high/mid/low CV) | Generic factual + bull/bear/neutral across volatility levels |
| **3E** | What is the practical value of summary quality? | S&P 500 (with ground truth) | Quality ladder: LLM-generated → expert → generic → random → empty |

In [ ]:
import sys, os
sys.path.insert(0, "synthefy-migas/src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from migaseval import MigasPipeline
from experiment_utils import (
    MigasInternalsExtractor,
    GENERIC_FACTUAL,
    load_financial_data,
    load_cement_skus,
    describe_series,
    build_summary,
    text_shift,
    directional_accuracy,
    plot_convex_weights,
    plot_forecast_comparison,
    plot_heatmap,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 100

In [ ]:
pipeline = MigasPipeline.from_pretrained("Synthefy/migas-1.5", device="cpu")
extractor = MigasInternalsExtractor(pipeline)

PRED_LEN = 8

## Data Loading

Download weekly close prices for 3 financial assets (model's native domain) and load the cement dataset (non-financial, domain transfer test).

In [ ]:
# Financial data: S&P 500, Gold ETF, Bitcoin
TICKERS = {"S&P 500": "^GSPC", "Gold (GLD)": "GLD", "Bitcoin": "BTC-USD"}
financial_data = {}

for name, ticker in TICKERS.items():
    values, dates = load_financial_data(ticker, period="2y", interval="1wk")
    # Split: last 8 weeks as ground truth
    ctx = values[:-PRED_LEN]
    gt = values[-PRED_LEN:]
    financial_data[name] = {"values": values, "ctx": ctx, "gt": gt, "dates": dates}
    print(f"{name}: {len(values)} weeks, ctx={len(ctx)}, "
          f"mean={ctx.mean():,.1f}, std={ctx.std():,.1f}, CV={ctx.std()/ctx.mean():.3f}")

# Cement data: same SKU as NB1
cement_full = pd.read_csv("data/master_12Mar26_top20_skus_apr2025_onwards_weekly.csv")
cement_full["date"] = pd.to_datetime(cement_full["date"])
stats_all = (
    cement_full.groupby(["sku", "SKU Name"])["qty"]
    .agg(["mean", "std"]).assign(cv=lambda x: x["std"] / x["mean"])
    .sort_values("cv", ascending=False)
)
CEMENT_SKU = stats_all.index[0][0]
CEMENT_NAME = stats_all.index[0][1]
sku_df = cement_full[cement_full["sku"] == CEMENT_SKU].sort_values("date").reset_index(drop=True)
cement_ctx = sku_df["qty"].values[:-PRED_LEN].astype(np.float32)
cement_gt = sku_df["qty"].values[-PRED_LEN:].astype(np.float32)
print(f"\nCement ({CEMENT_SKU}): {len(sku_df)} weeks, ctx={len(cement_ctx)}, "
      f"mean={cement_ctx.mean():,.0f}, CV={cement_ctx.std()/cement_ctx.mean():.3f}")

In [ ]:
# Visualize all 4 context series
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, (name, d) in zip(axes.flat[:3], financial_data.items()):
    ax.plot(d["ctx"], "o-", ms=2, color="steelblue", label="Context")
    t_gt = range(len(d["ctx"]), len(d["ctx"]) + PRED_LEN)
    ax.plot(t_gt, d["gt"], "o-", ms=4, color="black", label="Ground Truth")
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.set_ylabel("Price")

ax = axes.flat[3]
ax.plot(cement_ctx, "o-", ms=2, color="steelblue", label="Context")
t_gt = range(len(cement_ctx), len(cement_ctx) + PRED_LEN)
ax.plot(t_gt, cement_gt, "o-", ms=4, color="black", label="Ground Truth")
ax.set_title(f"Cement ({CEMENT_SKU[:12]}...)")
ax.legend(fontsize=8)
ax.set_ylabel("Dispatch Qty")

plt.suptitle("Context Series + Ground Truth", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Shared Prompt Definitions

Three predictive signal variants used across experiments:

In [ ]:
PREDICTIVE = {
    "bullish": (
        "Strong upward momentum expected. Multiple positive catalysts converging. "
        "Likely to see 10-20% appreciation over the forecast horizon."
    ),
    "bearish": (
        "Significant decline expected. Headwinds intensifying and momentum fading. "
        "Risk of 10-20% pullback over the forecast horizon."
    ),
    "neutral": (
        "No strong directional signals. Expect continuation of recent patterns "
        "with typical fluctuations around current levels."
    ),
}

print("Predictive signals:")
for k, v in PREDICTIVE.items():
    print(f"  {k}: {v[:60]}...")
print(f"\nGeneric factual: '{GENERIC_FACTUAL}'")

## Experiment 3A — The Factual Bottleneck on Real Financial Data

Test {Rich, Generic} × {Bullish, Bearish, Neutral} = **6 scenarios per asset** on S&P 500, Gold, and Bitcoin.

**Key questions:**
1. Does generic factual produce lower `w` than rich factual on real data? (validates NB1)
2. Is bull-bear separation larger with generic factual? (confirms the bottleneck)

In [ ]:
# Run 6 scenarios per asset
results_3a = {}  # (asset, factual_type, pred_type) -> results

for asset_name, d in financial_data.items():
    ctx = d["ctx"]
    rich_fact = describe_series(ctx)
    for fact_type, fact_text in [("Rich", rich_fact), ("Generic", GENERIC_FACTUAL)]:
        for pred_type, pred_text in PREDICTIVE.items():
            summary = build_summary(fact_text, pred_text)
            key = (asset_name, fact_type, pred_type)
            results_3a[key] = extractor.run(ctx, summary, pred_len=PRED_LEN)
    print(f"  {asset_name}: done (6 scenarios)")

print(f"\nTotal runs: {len(results_3a)}")

In [ ]:
# Build summary table: w and text_shift for each condition
rows = []
for asset_name in financial_data:
    chronos_fc = results_3a[(asset_name, "Rich", "bullish")]["chronos_forecast"]
    gt = financial_data[asset_name]["gt"]
    for fact_type in ["Rich", "Generic"]:
        for pred_type in PREDICTIVE:
            r = results_3a[(asset_name, fact_type, pred_type)]
            rows.append({
                "Asset": asset_name,
                "Factual": fact_type,
                "Predictive": pred_type,
                "Mean w": r["convex_weights"].mean(),
                "Text Shift": text_shift(r["forecast"], chronos_fc),
                "MAE vs GT": np.mean(np.abs(r["forecast"] - gt)),
            })

df_3a = pd.DataFrame(rows)
print(df_3a.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Key comparison: Rich vs Generic factual (mean w and bull-bear spread)
print("═══ Factual Bottleneck Test ═══\n")
print(f"{'Asset':<15} {'Metric':<20} {'Rich':>10} {'Generic':>10} {'Δ':>10}")
print("-" * 68)

for asset_name in financial_data:
    chronos_fc = results_3a[(asset_name, "Rich", "bullish")]["chronos_forecast"]

    # Mean w
    w_rich = np.mean([results_3a[(asset_name, "Rich", p)]["convex_weights"].mean() for p in PREDICTIVE])
    w_gen = np.mean([results_3a[(asset_name, "Generic", p)]["convex_weights"].mean() for p in PREDICTIVE])
    print(f"{asset_name:<15} {'Mean w':<20} {w_rich:>10.4f} {w_gen:>10.4f} {w_gen - w_rich:>+10.4f}")

    # Bull-Bear MAE (predictive influence)
    for ft in ["Rich", "Generic"]:
        fc_bull = results_3a[(asset_name, ft, "bullish")]["forecast"]
        fc_bear = results_3a[(asset_name, ft, "bearish")]["forecast"]
        bb = np.mean(np.abs(fc_bull - fc_bear))
        if ft == "Rich":
            bb_rich = bb
        else:
            bb_gen = bb
    print(f"{'':<15} {'Bull-Bear MAE':<20} {bb_rich:>10.1f} {bb_gen:>10.1f} {bb_gen - bb_rich:>+10.1f}")

    # Mean text shift
    ts_rich = np.mean([text_shift(results_3a[(asset_name, "Rich", p)]["forecast"], chronos_fc) for p in PREDICTIVE])
    ts_gen = np.mean([text_shift(results_3a[(asset_name, "Generic", p)]["forecast"], chronos_fc) for p in PREDICTIVE])
    print(f"{'':<15} {'Mean Text Shift':<20} {ts_rich:>10.1f} {ts_gen:>10.1f} {ts_gen - ts_rich:>+10.1f}")
    print()

In [ ]:
# Visualization: forecasts for each asset (Rich vs Generic × Bull vs Bear)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, asset_name in zip(axes, financial_data):
    d = financial_data[asset_name]
    ctx, gt = d["ctx"], d["gt"]
    chronos_fc = results_3a[(asset_name, "Rich", "bullish")]["chronos_forecast"]
    t_fc = range(len(ctx), len(ctx) + PRED_LEN)

    ax.plot(list(range(len(ctx)))[-12:], ctx[-12:], "o-", color="steelblue", ms=3, label="Context")
    ax.plot(t_fc, gt, "o-", color="black", ms=5, lw=2, label="Ground Truth")
    ax.plot(t_fc, chronos_fc, "s--", color="gray", ms=4, lw=2, label="Chronos-2")

    styles = {
        ("Rich", "bullish"): ("tab:red", "--", "Rich+Bull"),
        ("Rich", "bearish"): ("tab:blue", "--", "Rich+Bear"),
        ("Generic", "bullish"): ("tab:red", "-", "Gen+Bull"),
        ("Generic", "bearish"): ("tab:blue", "-", "Gen+Bear"),
    }
    for (ft, pt), (color, ls, label) in styles.items():
        fc = results_3a[(asset_name, ft, pt)]["forecast"]
        ax.plot(t_fc, fc, f"s{ls}", color=color, ms=3, alpha=0.8, label=label)

    ax.set_title(asset_name, fontsize=11)
    ax.legend(fontsize=7, loc="best")
    ax.set_ylabel("Value")

plt.suptitle("Exp 3A: Rich vs Generic Factual — Bullish vs Bearish", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Experiment 3B — Can Predictive Signals Steer Direction?

Using **only generic factual** (to remove the bottleneck), test whether bullish vs bearish predictive signals produce meaningfully different forecast trajectories across all 3 financial assets.

**Key question:** With the factual bottleneck removed, do predictive signals produce meaningful directional shifts on the model's native (financial) domain?

In [ ]:
# Directional analysis: for each asset, compare Chronos, Generic+Bull, Generic+Bear
print("═══ Directional Steering Test (Generic Factual Only) ═══\n")
print(f"{'Asset':<15} {'Condition':<15} {'Mean FC':>12} {'Δ vs Chronos':>14} {'Dir Acc':>10} {'Text Shift':>12}")
print("-" * 80)

for asset_name in financial_data:
    gt = financial_data[asset_name]["gt"]
    chronos_fc = results_3a[(asset_name, "Rich", "bullish")]["chronos_forecast"]
    chronos_mean = chronos_fc.mean()

    da_chronos = directional_accuracy(chronos_fc, gt)
    print(f"{asset_name:<15} {'Chronos-2':<15} {chronos_mean:>12,.1f} {'—':>14} {da_chronos:>10.2f} {'—':>12}")

    for pred_type in ["bullish", "bearish", "neutral"]:
        r = results_3a[(asset_name, "Generic", pred_type)]
        fc = r["forecast"]
        shift = text_shift(fc, chronos_fc)
        delta = fc.mean() - chronos_mean
        da = directional_accuracy(fc, gt)
        print(f"{'':<15} {'Gen+' + pred_type:<15} {fc.mean():>12,.1f} {delta:>+14,.1f} {da:>10.2f} {shift:>12.1f}")
    print()

In [ ]:
# Convex weight comparison: Generic factual across all assets and predictive signals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, asset_name in zip(axes, financial_data):
    steps = np.arange(PRED_LEN)
    for pred_type in PREDICTIVE:
        w = results_3a[(asset_name, "Generic", pred_type)]["convex_weights"]
        ax.plot(steps, w, "o-", ms=4, label=f"Gen+{pred_type}")
    # Also show Rich+neutral for comparison
    w_rich = results_3a[(asset_name, "Rich", "neutral")]["convex_weights"]
    ax.plot(steps, w_rich, "s--", ms=4, color="gray", alpha=0.7, label="Rich+neutral")
    ax.set_title(asset_name, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="gray", ls="--", alpha=0.3)
    ax.set_xlabel("Step")
    ax.set_ylabel("w")
    ax.legend(fontsize=7)

plt.suptitle("Exp 3B: Convex Weights — Generic Factual × Predictive Signals", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Experiment 3C — Cement Data: Confound Fixed

The original experiment used **rich factual for bull/bear** but **generic factual for neutral**, creating a confound. Now we test:

1. **All Rich factual** + {Bull, Bear, Neutral} — isolate predictive signal effect
2. **All Generic factual** + {Bull, Bear, Neutral} — maximize text influence

**Key question:** With the confound fixed, how much do predictive signals alone move the cement forecast?

In [ ]:
# Cement confound test
rich_fact_cement = describe_series(cement_ctx)

results_3c = {}
for fact_type, fact_text in [("Rich", rich_fact_cement), ("Generic", GENERIC_FACTUAL)]:
    for pred_type, pred_text in PREDICTIVE.items():
        summary = build_summary(fact_text, pred_text)
        results_3c[(fact_type, pred_type)] = extractor.run(cement_ctx, summary, pred_len=PRED_LEN)

chronos_fc = results_3c[("Rich", "bullish")]["chronos_forecast"]

print(f"{'Factual':<10} {'Predictive':<12} {'Mean w':>10} {'Text Shift':>12} {'MAE vs GT':>12} {'Bull-Bear':>12}")
print("-" * 70)
for fact_type in ["Rich", "Generic"]:
    fc_bull = results_3c[(fact_type, "bullish")]["forecast"]
    fc_bear = results_3c[(fact_type, "bearish")]["forecast"]
    bb_mae = np.mean(np.abs(fc_bull - fc_bear))
    for pred_type in PREDICTIVE:
        r = results_3c[(fact_type, pred_type)]
        w = r["convex_weights"].mean()
        shift = text_shift(r["forecast"], chronos_fc)
        mae = np.mean(np.abs(r["forecast"] - cement_gt))
        bb_str = f"{bb_mae:,.0f}" if pred_type == "bullish" else ""
        print(f"{fact_type:<10} {pred_type:<12} {w:>10.4f} {shift:>12,.0f} {mae:>12,.0f} {bb_str:>12}")
    print()

In [ ]:
# Visualize: Rich vs Generic factual on cement
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
t_fc = range(len(cement_ctx), len(cement_ctx) + PRED_LEN)

for ax, fact_type in zip(axes, ["Rich", "Generic"]):
    ax.plot(list(range(len(cement_ctx)))[-12:], cement_ctx[-12:], "o-",
            color="steelblue", ms=3, label="Context")
    ax.plot(t_fc, cement_gt, "o-", color="black", ms=5, lw=2, label="Ground Truth")
    ax.plot(t_fc, chronos_fc, "s--", color="gray", ms=4, lw=2, label="Chronos-2")

    for pred_type, color in [("bullish", "tab:red"), ("bearish", "tab:blue"), ("neutral", "tab:green")]:
        fc = results_3c[(fact_type, pred_type)]["forecast"]
        ax.plot(t_fc, fc, "s--", color=color, ms=3, alpha=0.8, label=pred_type)

    ax.set_title(f"{fact_type} Factual (Confound Fixed)", fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylabel("Dispatch Qty")

plt.suptitle("Exp 3C: Cement — Confound Fixed", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Experiment 3D — Multi-SKU Volatility Test

Pick 3 cement SKUs with high, medium, and low CV. Use **generic factual** + {Bull, Bear, Neutral} for each.

**Key question:** Does real-data volatility affect text sensitivity? (validates NB2 synthetic finding that trend-down was most sensitive, constant was immune)

In [ ]:
# Load 3 SKUs
multi_sku_df, sku_stats = load_cement_skus(n_skus=3)

sku_data = {}
for sku_id in multi_sku_df["sku"].unique():
    sub = multi_sku_df[multi_sku_df["sku"] == sku_id].sort_values("date").reset_index(drop=True)
    cv = sub["cv"].iloc[0]
    sku_name = sub["SKU Name"].iloc[0]
    ctx = sub["qty"].values[:-PRED_LEN].astype(np.float32)
    gt_vals = sub["qty"].values[-PRED_LEN:].astype(np.float32)
    label = f"{sku_name[:20]}... (CV={cv:.2f})"
    sku_data[label] = {"ctx": ctx, "gt": gt_vals, "sku": sku_id, "cv": cv}

for label, d in sku_data.items():
    print(f"{label}: ctx={len(d['ctx'])}, mean={d['ctx'].mean():,.0f}")

In [ ]:
# Run Generic factual × 3 predictive signals for each SKU
results_3d = {}
for label, d in sku_data.items():
    for pred_type, pred_text in PREDICTIVE.items():
        summary = build_summary(GENERIC_FACTUAL, pred_text)
        results_3d[(label, pred_type)] = extractor.run(d["ctx"], summary, pred_len=PRED_LEN)
    print(f"  {label}: done")

# Build comparison table
print(f"\n{'SKU':<35} {'Pred':<10} {'Mean w':>10} {'Text Shift':>12} {'MAE vs GT':>12}")
print("-" * 82)
for label, d in sku_data.items():
    chronos_fc = results_3d[(label, "bullish")]["chronos_forecast"]
    for pred_type in PREDICTIVE:
        r = results_3d[(label, pred_type)]
        w = r["convex_weights"].mean()
        shift = text_shift(r["forecast"], chronos_fc)
        mae = np.mean(np.abs(r["forecast"] - d["gt"]))
        print(f"{label:<35} {pred_type:<10} {w:>10.4f} {shift:>12,.1f} {mae:>12,.0f}")
    print()

In [ ]:
# Visualize: text_shift and w by SKU volatility
labels = list(sku_data.keys())
cvs = [sku_data[l]["cv"] for l in labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(labels))
width = 0.25
for i, pred_type in enumerate(PREDICTIVE):
    shifts = []
    weights = []
    for label in labels:
        chronos_fc = results_3d[(label, "bullish")]["chronos_forecast"]
        r = results_3d[(label, pred_type)]
        shifts.append(text_shift(r["forecast"], chronos_fc))
        weights.append(r["convex_weights"].mean())
    ax1.bar(x + i * width, shifts, width, label=pred_type, alpha=0.8)
    ax2.bar(x + i * width, weights, width, label=pred_type, alpha=0.8)

ax1.set_xticks(x + width)
ax1.set_xticklabels([f"CV={cv:.2f}" for cv in cvs], rotation=15)
ax1.set_ylabel("Text Shift")
ax1.set_title("Text Shift by SKU Volatility")
ax1.legend(fontsize=8)

ax2.set_xticks(x + width)
ax2.set_xticklabels([f"CV={cv:.2f}" for cv in cvs], rotation=15)
ax2.set_ylabel("Mean w")
ax2.set_ylim(0, 1.05)
ax2.axhline(0.5, color="gray", ls="--", alpha=0.3)
ax2.set_title("Chronos Trust by SKU Volatility")
ax2.legend(fontsize=8)

plt.suptitle("Exp 3D: Multi-SKU Volatility Test (Generic Factual)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Experiment 3E — Summary Quality Ladder

For S&P 500, test a ladder of summary quality levels **with generic factual** (to maximize text influence):

| Level | Factual | Predictive |
|-------|---------|------------|
| **Expert** | Generic | Hand-crafted detailed analysis |
| **Bullish** | Generic | Standard bullish signal |
| **Bearish** | Generic | Standard bearish signal |
| **Neutral** | Generic | Standard neutral signal |
| **Vague** | Generic | Very vague ("things may change") |
| **Random** | Random gibberish | Random gibberish |
| **Empty** | Empty | Empty |

**Key question:** Does summary quality affect forecast accuracy (MAE vs ground truth)?

In [ ]:
# Quality ladder on S&P 500
sp_ctx = financial_data["S&P 500"]["ctx"]
sp_gt = financial_data["S&P 500"]["gt"]

QUALITY_LADDER = {
    "Expert": build_summary(
        GENERIC_FACTUAL,
        "Based on recent price action and macro indicators, the index shows strong "
        "support at current levels with positive breadth divergence. The Federal Reserve's "
        "stance suggests continued accommodative conditions. Earnings season momentum "
        "and institutional positioning favor moderate upside of 3-5% over the next 8 weeks, "
        "with key risk events around mid-period that could cause temporary pullbacks."
    ),
    "Bullish": build_summary(GENERIC_FACTUAL, PREDICTIVE["bullish"]),
    "Bearish": build_summary(GENERIC_FACTUAL, PREDICTIVE["bearish"]),
    "Neutral": build_summary(GENERIC_FACTUAL, PREDICTIVE["neutral"]),
    "Vague": build_summary(GENERIC_FACTUAL, "Things may or may not change."),
    "Random": build_summary("xyzzy foo bar quux blargh", "asdf qwerty zxcv wombat defenestrate"),
    "Empty": build_summary("", ""),
}

results_3e = {}
for quality, summary in QUALITY_LADDER.items():
    results_3e[quality] = extractor.run(sp_ctx, summary, pred_len=PRED_LEN)

chronos_fc = results_3e["Expert"]["chronos_forecast"]

In [ ]:
# Forecast comparison
ladder_forecasts = {q: r["forecast"] for q, r in results_3e.items()}
ladder_forecasts["Chronos-2"] = chronos_fc

fig, ax = plot_forecast_comparison(
    sp_ctx, ladder_forecasts, ground_truth=sp_gt,
    title="Exp 3E: Summary Quality Ladder — S&P 500"
)
plt.show()

# Metrics table
print(f"{'Quality':<12} {'MAE vs GT':>12} {'Text Shift':>12} {'Mean w':>10} {'Dir Acc':>10}")
print("-" * 58)
for quality, r in results_3e.items():
    mae = np.mean(np.abs(r["forecast"] - sp_gt))
    shift = text_shift(r["forecast"], chronos_fc)
    da = directional_accuracy(r["forecast"], sp_gt)
    print(f"{quality:<12} {mae:>12,.1f} {shift:>12.1f} {r['convex_weights'].mean():>10.4f} {da:>10.2f}")

# Chronos baseline
mae_ch = np.mean(np.abs(chronos_fc - sp_gt))
da_ch = directional_accuracy(chronos_fc, sp_gt)
print(f"{'Chronos-2':<12} {mae_ch:>12,.1f} {'—':>12} {'—':>10} {da_ch:>10.2f}")

In [ ]:
# Bar chart: MAE vs GT for each quality level
qualities = list(results_3e.keys())
maes = [np.mean(np.abs(results_3e[q]["forecast"] - sp_gt)) for q in qualities]
maes.append(np.mean(np.abs(chronos_fc - sp_gt)))
qualities.append("Chronos-2")
mean_ws = [results_3e[q]["convex_weights"].mean() for q in qualities[:-1]]
mean_ws.append(1.0)  # Chronos is w=1 by definition

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ["tab:green"] * 4 + ["tab:orange"] * 2 + ["tab:red"] + ["gray"]
ax1.bar(range(len(qualities)), maes, color=colors, alpha=0.8)
ax1.set_xticks(range(len(qualities)))
ax1.set_xticklabels(qualities, rotation=30, ha="right")
ax1.set_ylabel("MAE vs Ground Truth")
ax1.set_title("Forecast Accuracy by Summary Quality")

ax2.bar(range(len(qualities)), mean_ws, color=colors, alpha=0.8)
ax2.set_xticks(range(len(qualities)))
ax2.set_xticklabels(qualities, rotation=30, ha="right")
ax2.set_ylabel("Mean Convex Weight (w)")
ax2.set_ylim(0, 1.05)
ax2.axhline(0.5, color="gray", ls="--", alpha=0.3)
ax2.set_title("Chronos Trust by Summary Quality")

plt.suptitle("Exp 3E: Quality Ladder — MAE and Convex Weights", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Summary of Key Findings

<!-- Fill in after running:

### Exp 3A — Factual Bottleneck
- **Does generic factual lower w on real data?** ___
- **Is bull-bear separation larger with generic factual?** ___
- **Magnitude of effect on financial vs NB1 cement data:** ___

### Exp 3B — Predictive Signal Steering
- **Do bullish/bearish forecasts differ meaningfully with generic factual?** ___
- **Does the direction match (bullish → higher, bearish → lower)?** ___
- **Does text conditioning improve directional accuracy vs Chronos?** ___

### Exp 3C — Cement Confound Fix
- **Bull-Bear MAE with Rich factual (confound removed):** ___
- **Bull-Bear MAE with Generic factual:** ___
- **Confirms NB1 finding?** ___

### Exp 3D — Volatility Effect
- **Does higher CV → more text sensitivity?** ___
- **Consistent with NB2 synthetic findings?** ___

### Exp 3E — Quality Ladder
- **Does better summary quality → lower MAE?** ___
- **Best performing summary type:** ___
- **Does empty/random text hurt or help vs Chronos baseline?** ___
- **Practical recommendation for summary generation:** ___
-->